# Train on a Colab GPU

Runs `scripts/train.py` on a Colab GPU runtime instead of the local CPU. It clones the repo,
downloads the PhiUSIIL dataset (UCI) and the Tranco list, builds the splits, trains and evaluates.

Connect this notebook to a Colab **GPU** runtime first, either in the VS Code Colab extension or
in the Colab web UI under *Runtime → Change runtime type*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU type"
print(torch.cuda.get_device_name(0))

In [ ]:
REPO = "https://github.com/Kluey/URL-Phishing-Classifier.git"
BRANCH = "hostname-input"

%cd /content
!rm -rf URL-Phishing-Classifier && git clone -q --depth 1 -b {BRANCH} {REPO}
%cd /content/URL-Phishing-Classifier
!git log --oneline -1
!pip install -q -e ".[pretrain]"

In [ ]:
!mkdir -p data model
!wget -q -O /tmp/phiusiil.zip "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
!unzip -o -q /tmp/phiusiil.zip -d data && ls -la data

Builds the hostname splits (PhiUSIIL + Tranco subdomains, all balanced data) and trains for 2 epochs.

In [ ]:
!python scripts/train.py --prepare-data 2>&1 | tee model/train.log

In [ ]:
!python scripts/evaluate.py --split test
!python scripts/evaluate.py --split real_world

Files to bring back, each into the same folder in the local project. In VS Code, use the Colab
sidebar → **Contents** → right-click a file → **Download...**. The split checksums show whether
the local splits are identical; if they are, only the `model/` files are needed.

In [ ]:
!cd /content/URL-Phishing-Classifier && md5sum data/train.csv data/validation.csv data/test.csv && ls -la model/